Hier werden nun die ICON-D2 Daten gewodnloadet. Und zwar: Wind Direction (in u/v), Wind_Speed, Precipitation und Temp (2m), wobei Precipitation für 12 Std kumuliert aufgenommen wird und alle anderen den Wert von 13 Uhr übernehmen.

In [1]:
import os
import time
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
import requests
from scipy.interpolate import griddata

# =========================================================
# 1. Einstellungen & Pfade
# =========================================================
OUTPUT_DIR = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\New_ICON_D2_Rasters"
os.makedirs(os.path.join(OUTPUT_DIR, "Temp_2m"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "Wind_Speed"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "U_Wind"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "V_Wind"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "Precipitation_12h"), exist_ok=True) # Neuer Ordnername!

START_DATE = "2024-05-01"
END_DATE = "2024-08-31"

LON_MIN, LON_MAX = 9.85, 10.05
LAT_MIN, LAT_MAX = 49.74, 49.85

lons = np.arange(LON_MIN, LON_MAX, 0.02)
lats = np.arange(LAT_MIN, LAT_MAX, 0.02)
grid_lons, grid_lats = np.meshgrid(lons, lats)
points_wgs84 = pd.DataFrame(
    {"lon": grid_lons.flatten(), "lat": grid_lats.flatten()}
)

print(f"Erzeuge DWD Gitter-Netzwerk für Würzburg mit {len(points_wgs84)} Modellpunkten...")

# =========================================================
# 2. Wetterdaten abrufen & INTELLIGENT filtern
# =========================================================
all_grid_data = []

for idx, row in points_wgs84.iterrows():
    lat, lon = row["lat"], row["lon"]
    print(f" -> Lade Gitterpunkt {idx+1}/{len(points_wgs84)} ({lat:.4f}, {lon:.4f})...")

    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ["temperature_2m", "wind_speed_10m", "wind_direction_10m", "precipitation"],
        "models": "icon_d2",
        "timezone": "Europe/Berlin",
    }

    try:
        res = requests.get(url, params=params).json()
        if "error" in res:
            continue

        df_pt = pd.DataFrame({
            "time": pd.to_datetime(res["hourly"]["time"]),
            "temp": res["hourly"]["temperature_2m"],
            "wind_kmh": res["hourly"]["wind_speed_10m"],
            "wind_dir": res["hourly"]["wind_direction_10m"],
            "precip": res["hourly"]["precipitation"],
        })

        # Hilfsspalten für Datum und Stunde
        df_pt["date"] = df_pt["time"].dt.date
        df_pt["hour"] = df_pt["time"].dt.hour

        # --- A) Snapshot für 13:00 Uhr (Temp & Wind) ---
        df_13 = df_pt[df_pt["hour"] == 13].copy()
        df_13["wind_ms"] = df_13["wind_kmh"] / 3.6
        rads = np.radians(df_13["wind_dir"])
        df_13["u_10"] = -df_13["wind_ms"] * np.sin(rads)
        df_13["v_10"] = -df_13["wind_ms"] * np.cos(rads)

        # --- B) 12-Stunden-Kumulation (Niederschlag 01:00 bis 13:00 Uhr) ---
        # Wir filtern die Stunden 1 bis 13, gruppieren nach Tag und bilden die Summe
        df_precip = df_pt[(df_pt["hour"] >= 1) & (df_pt["hour"] <= 13)]
        precip_sum = df_precip.groupby("date")["precip"].sum().reset_index()
        precip_sum.rename(columns={"precip": "precip_12h_sum"}, inplace=True)

        # --- C) Zusammenführen ---
        df_daily = pd.merge(df_13, precip_sum, on="date", how="left")
        df_daily["lat"] = lat
        df_daily["lon"] = lon

        all_grid_data.append(df_daily)

    except Exception as e:
        print(f"Fehler: {e}")
    time.sleep(0.3)

df_all = pd.concat(all_grid_data, ignore_index=True)

# =========================================================
# 3. Geo-Transformation in DOM-Koordinaten (EPSG:25832)
# =========================================================
print("\nTransformiere Gitter in metrisches System (UTM 32N / EPSG:25832)...")
gdf_all = gpd.GeoDataFrame(
    df_all,
    geometry=gpd.points_from_xy(df_all["lon"], df_all["lat"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")

gdf_all["x"] = gdf_all.geometry.x
gdf_all["y"] = gdf_all.geometry.y

# =========================================================
# 4. Raster-Parameter einstellen
# =========================================================
X_MIN, X_MAX = 560000.0, 575000.0
Y_MIN, Y_MAX = 5510000.0, 5522000.0
PIXEL_SIZE = 100.0

width = int((X_MAX - X_MIN) / PIXEL_SIZE)
height = int((Y_MAX - Y_MIN) / PIXEL_SIZE)
transform = from_origin(X_MIN, Y_MAX, PIXEL_SIZE, PIXEL_SIZE)

grid_x, grid_y = np.meshgrid(
    np.linspace(X_MIN, X_MAX, width), np.linspace(Y_MAX, Y_MIN, height)
)

# =========================================================
# 5. Tägliche GeoTIFFs generieren
# =========================================================
unique_days = gdf_all["time"].dt.strftime("%Y-%m-%d").unique()
print(f"\nGeneriere {len(unique_days)} tägliche GeoTIFFs pro Parameter...")

for day in unique_days:
    df_day = gdf_all[gdf_all["time"].dt.strftime("%Y-%m-%d") == day]
    points = df_day[["x", "y"]].values

    # NEU: Das kumulierte 'precip_12h_sum' wird anstelle von 'precip' verwendet
    params_to_grid = {
        "Temp_2m": ("temp", f"ICON_D2_Temp_{day}.tif"),
        "Wind_Speed": ("wind_ms", f"ICON_D2_WindSpeed_{day}.tif"),
        "U_Wind": ("u_10", f"ICON_D2_U_{day}.tif"),
        "V_Wind": ("v_10", f"ICON_D2_V_{day}.tif"),
        "Precipitation_12h": ("precip_12h_sum", f"ICON_D2_Precip_12h_{day}.tif"),
    }

    for folder, (col, filename) in params_to_grid.items():
        values = df_day[col].values

        raster_array = griddata(
            points, values, (grid_x, grid_y), method="nearest"
        )

        out_path = os.path.join(OUTPUT_DIR, folder, filename)

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype=raster_array.dtype,
            crs="EPSG:25832",
            transform=transform,
        ) as dst:
            dst.write(raster_array, 1)

print(f"\nFERTIG! Alle Rasterdateien liegen bereit unter:\n{OUTPUT_DIR}")

Erzeuge DWD Gitter-Netzwerk für Würzburg mit 66 Modellpunkten...
 -> Lade Gitterpunkt 1/66 (49.7400, 9.8500)...
 -> Lade Gitterpunkt 2/66 (49.7400, 9.8700)...
 -> Lade Gitterpunkt 3/66 (49.7400, 9.8900)...
 -> Lade Gitterpunkt 4/66 (49.7400, 9.9100)...
 -> Lade Gitterpunkt 5/66 (49.7400, 9.9300)...
 -> Lade Gitterpunkt 6/66 (49.7400, 9.9500)...
 -> Lade Gitterpunkt 7/66 (49.7400, 9.9700)...
 -> Lade Gitterpunkt 8/66 (49.7400, 9.9900)...
 -> Lade Gitterpunkt 9/66 (49.7400, 10.0100)...
 -> Lade Gitterpunkt 10/66 (49.7400, 10.0300)...
 -> Lade Gitterpunkt 11/66 (49.7400, 10.0500)...
 -> Lade Gitterpunkt 12/66 (49.7600, 9.8500)...
 -> Lade Gitterpunkt 13/66 (49.7600, 9.8700)...
 -> Lade Gitterpunkt 14/66 (49.7600, 9.8900)...
 -> Lade Gitterpunkt 15/66 (49.7600, 9.9100)...
 -> Lade Gitterpunkt 16/66 (49.7600, 9.9300)...
 -> Lade Gitterpunkt 17/66 (49.7600, 9.9500)...
 -> Lade Gitterpunkt 18/66 (49.7600, 9.9700)...
 -> Lade Gitterpunkt 19/66 (49.7600, 9.9900)...
 -> Lade Gitterpunkt 20/66 (4

Hier werden nun noch die Relative humidity (2m) ICON-D2 Daten gewodnloadet und in einen neuen Ordner gepackt.

In [1]:
import os
import time
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
import requests
from scipy.interpolate import griddata

# =========================================================
# 1. Einstellungen & Pfade
# =========================================================
OUTPUT_DIR = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\New_ICON_D2_Rasters"

# Nur der neue Ordner für die Luftfeuchtigkeit wird erstellt
os.makedirs(os.path.join(OUTPUT_DIR, "Relative_Humidity_2m"), exist_ok=True)

START_DATE = "2024-05-01"
END_DATE = "2024-08-31"

LON_MIN, LON_MAX = 9.85, 10.05
LAT_MIN, LAT_MAX = 49.74, 49.85

lons = np.arange(LON_MIN, LON_MAX, 0.02)
lats = np.arange(LAT_MIN, LAT_MAX, 0.02)
grid_lons, grid_lats = np.meshgrid(lons, lats)
points_wgs84 = pd.DataFrame(
    {"lon": grid_lons.flatten(), "lat": grid_lats.flatten()}
)

print(f"Erzeuge DWD Gitter-Netzwerk für Würzburg mit {len(points_wgs84)} Modellpunkten...")

# =========================================================
# 2. Wetterdaten abrufen (Nur Relative Humidity)
# =========================================================
all_grid_data = []

for idx, row in points_wgs84.iterrows():
    lat, lon = row["lat"], row["lon"]
    print(f" -> Lade Gitterpunkt {idx+1}/{len(points_wgs84)} ({lat:.4f}, {lon:.4f})...")

    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ["relative_humidity_2m"], # Nur dieser Parameter wird geladen
        "models": "icon_d2",
        "timezone": "Europe/Berlin",
    }

    try:
        res = requests.get(url, params=params).json()
        if "error" in res:
            print(f"    API-Fehler am Punkt {lat}, {lon}")
            continue

        df_pt = pd.DataFrame({
            "time": pd.to_datetime(res["hourly"]["time"]),
            "humidity": res["hourly"]["relative_humidity_2m"]
        })

        # Hilfsspalten für Datum und Stunde
        df_pt["date"] = df_pt["time"].dt.date
        df_pt["hour"] = df_pt["time"].dt.hour

        # --- Snapshot für 13:00 Uhr herausfiltern ---
        df_13 = df_pt[df_pt["hour"] == 13].copy()

        # Koordinaten für die spätere Interpolation anheften
        df_13["lat"] = lat
        df_13["lon"] = lon

        all_grid_data.append(df_13)

    except Exception as e:
        print(f"Fehler bei Anfrage: {e}")

    # Kurze Pause, um die API nicht zu überlasten
    time.sleep(0.3)

df_all = pd.concat(all_grid_data, ignore_index=True)

# =========================================================
# 3. Geo-Transformation in DOM-Koordinaten (EPSG:25832)
# =========================================================
print("\nTransformiere Gitter in metrisches System (UTM 32N / EPSG:25832)...")
gdf_all = gpd.GeoDataFrame(
    df_all,
    geometry=gpd.points_from_xy(df_all["lon"], df_all["lat"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")

gdf_all["x"] = gdf_all.geometry.x
gdf_all["y"] = gdf_all.geometry.y

# =========================================================
# 4. Raster-Parameter einstellen (100m Auflösung)
# =========================================================
X_MIN, X_MAX = 560000.0, 575000.0
Y_MIN, Y_MAX = 5510000.0, 5522000.0
PIXEL_SIZE = 100.0

width = int((X_MAX - X_MIN) / PIXEL_SIZE)
height = int((Y_MAX - Y_MIN) / PIXEL_SIZE)
transform = from_origin(X_MIN, Y_MAX, PIXEL_SIZE, PIXEL_SIZE)

grid_x, grid_y = np.meshgrid(
    np.linspace(X_MIN, X_MAX, width), np.linspace(Y_MAX, Y_MIN, height)
)

# =========================================================
# 5. Tägliche GeoTIFFs generieren
# =========================================================
unique_days = gdf_all["time"].dt.strftime("%Y-%m-%d").unique()
print(f"\nGeneriere {len(unique_days)} tägliche GeoTIFFs für die relative Luftfeuchtigkeit...")

for day in unique_days:
    df_day = gdf_all[gdf_all["time"].dt.strftime("%Y-%m-%d") == day]
    points = df_day[["x", "y"]].values
    values = df_day["humidity"].values

    # Interpolation der Punkte auf das 100x100m Raster (Nearest Neighbor)
    raster_array = griddata(
        points, values, (grid_x, grid_y), method="nearest"
    )

    out_path = os.path.join(OUTPUT_DIR, "Relative_Humidity_2m", f"ICON_D2_RelHumid_{day}.tif")

    with rasterio.open(
        out_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=raster_array.dtype,
        crs="EPSG:25832",
        transform=transform,
    ) as dst:
        dst.write(raster_array, 1)

print(f"\nFERTIG! Alle GeoTIFFs der Luftfeuchtigkeit liegen bereit unter:\n{os.path.join(OUTPUT_DIR, 'Relative_Humidity_2m')}")

Erzeuge DWD Gitter-Netzwerk für Würzburg mit 66 Modellpunkten...
 -> Lade Gitterpunkt 1/66 (49.7400, 9.8500)...
 -> Lade Gitterpunkt 2/66 (49.7400, 9.8700)...
 -> Lade Gitterpunkt 3/66 (49.7400, 9.8900)...
 -> Lade Gitterpunkt 4/66 (49.7400, 9.9100)...
 -> Lade Gitterpunkt 5/66 (49.7400, 9.9300)...
 -> Lade Gitterpunkt 6/66 (49.7400, 9.9500)...
 -> Lade Gitterpunkt 7/66 (49.7400, 9.9700)...
 -> Lade Gitterpunkt 8/66 (49.7400, 9.9900)...
 -> Lade Gitterpunkt 9/66 (49.7400, 10.0100)...
 -> Lade Gitterpunkt 10/66 (49.7400, 10.0300)...
 -> Lade Gitterpunkt 11/66 (49.7400, 10.0500)...
 -> Lade Gitterpunkt 12/66 (49.7600, 9.8500)...
 -> Lade Gitterpunkt 13/66 (49.7600, 9.8700)...
 -> Lade Gitterpunkt 14/66 (49.7600, 9.8900)...
 -> Lade Gitterpunkt 15/66 (49.7600, 9.9100)...
 -> Lade Gitterpunkt 16/66 (49.7600, 9.9300)...
 -> Lade Gitterpunkt 17/66 (49.7600, 9.9500)...
 -> Lade Gitterpunkt 18/66 (49.7600, 9.9700)...
 -> Lade Gitterpunkt 19/66 (49.7600, 9.9900)...
 -> Lade Gitterpunkt 20/66 (4